# Creating GAT Journey Planner Agent Model

### Read in training data -> NetworkX graph

In [47]:
import pandas as pd
import networkx as nx

data_folder_path = "C:\\Users\\sasab\\Documents\\Projects\\MaaS_AI\\Agents\\journey_planner\\data\\"

#pune_maas_journey_planner_data.csv - metro routing data table
route_edges_df = pd.read_csv(data_folder_path+"pune_maas_journey_planner_data.csv")

#encode 'string' value attributes (line,mode)
#for every new value map to unique 
line_map = {name: i for i, name in enumerate(route_edges_df["line"].unique())}     #line: purple | pink | aqua
mode_map = {name: i for i, name in enumerate(route_edges_df["mode"].unique(), start=1)} # mode: metro | feeder bus

route_edges_df["line_id"] = route_edges_df["line"].map(line_map)
route_edges_df["mode_id"] = route_edges_df["mode"].map(mode_map)
route_edges_df["is_transfer"] = route_edges_df["is_transfer"].astype(int)
route_edges_df["bidirectional"] = route_edges_df["bidirectional"].astype(int)

#timetable_data.csv - metro train timetable data
timetable_df = pd.read_csv(data_folder_path+"timetable_data.csv")

#swipe_metadata.csv - metro swipe data with each ticket pur
swipes_df = pd.read_csv(data_folder_path+"swipe_metadata.csv")

#check
# print ("========route_edges table========")
# print (route_edges_df)

# print ("\n========timetable table========")
# print (timetable_df)

# print ("\n========swipes metadata table========")
# print (swipes_df)

### Create a station master list

In [48]:
stations_from = route_edges_df[["station_id_from", "station_name_from"]].rename(
    columns={"station_id_from": "station_id", "station_name_from": "station_name"}
)

stations_to = route_edges_df[["station_id_to", "station_name_to"]].rename(
    columns={"station_id_to": "station_id", "station_name_to": "station_name"}
)

stations_df = pd.concat([stations_from, stations_to], ignore_index=True).drop_duplicates()
stations_df = stations_df.sort_values("station_id").reset_index(drop=True)

print(stations_df)

   station_id       station_name
0         A01      Chandni Chowk
1         A02  Kothrud Bus Depot
2         A03              Vanaz
3         A04        Anand Nagar
4         A05       Ideal Colony
..        ...                ...
70        P19        Market Yard
71        P20          Bibwewadi
72        P21          Padmavati
73        P22       Balaji Nagar
74        P23             Katraj

[75 rows x 2 columns]


In [ ]:
# define NetworkX graph
G = nx.DiGraph()

# Add node features
# mode_nodes = set()  #transport mode  
# transfer_nodes = set()  #transport mode 

#loop through rows and set edges
for _, row in route_edges_df.iterrows():
    G.add_node(
        row["station_id_from"],
        station_name = row["station_name_from"],
        mode_id=row["mode_id"],
        transfer=int(row["is_transfer"]),
        line_id=row["line_id"]
    )
    G.add_node(
        row["station_id_to"],
        station_name = row["station_name_to"],
        mode_id=row["mode_id"],
        transfer=int(row["is_transfer"]),
        line_id=row["line_id"]
    )
    
    G.add_edge(
        row["station_id_from"],
        row["station_id_to"],
        line_id=row["line_id"],
        length=float(row["distance_km"]),
        minutes=float(row["travel_time_min"]),
        transfer=int(row["is_transfer"]),
        transfer_time=float(row["transfer_penalty_min"])
    )

    if row["bidirectional"]:
        G.add_edge(
            row["station_id_to"],
            row["station_id_from"],
            line_id=row["line_id"],
            length=float(row["distance_km"]),
            minutes=float(row["travel_time_min"]),
            transfer=int(row["is_transfer"]),
            transfer_time=float(row["transfer_penalty_min"])
        )        
print(f"Nodes: {list(G.nodes)}")
print(f"Edges: {list(G.edges)}")


Nodes: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'K01', 'K02', 'K03', 'K04', 'K05', 'K06', 'K07', 'K08', 'K09', 'K10', 'K11', 'K12', 'K13', 'K14', 'K15', 'K16', 'K17', 'K18', 'K19', 'K20', 'K21', 'K22', 'K23']
Edges: [('P01', 'P02'), ('P02', 'P01'), ('P02', 'P03'), ('P03', 'P02'), ('P03', 'P04'), ('P04', 'P03'), ('P04', 'P05'), ('P05', 'P04'), ('P05', 'P06'), ('P06', 'P05'), ('P06', 'P07'), ('P07', 'P06'), ('P07', 'P08'), ('P08', 'P07'), ('P08', 'P09'), ('P09', 'P08'), ('P09', 'P10'), ('P10', 'P09'), ('P10', 'P11'), ('P11', 'P10'), ('P11', 'P12'), ('P12', 'P11'), ('P12', 'P13'), ('P13', 'P12'), ('P13', 'P14'), ('P14', 'P13'), ('P14', 'P15'), ('P14', 'K22'), ('P15', 'P1

In [ ]:
from torch_geometric.nn import GATv2Conv

class GAT(torch.nn.Module):
    # in_channels - number of node feature
    # hidden_channels - 16
    # out_channels - number of classes output channels.set to 3, will rep congestion level to use in Dispact Agent layer
    # edge_features - number of edge feature
    def __init__(self, in_channels, hidden_channels, out_channels,edge_features):
        super().__init__()
        # First GAT layer with multi-head attention
        # self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=8, edge_dim=edge_features)
        # Second GAT layer (output layer)
        # self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)
        self.conv2 = GATv2Conv(hidden_channels * 8,out_channels,heads=1,concat=False,edge_dim=edge_features)
        
    #dropout - default .6 or 60% once outr dataset is small and doent have alot of node feature
        #new default set to 0.1-0.2 for minimal regularization and small graphs
    def forward(self, x, edge_index, edge_attr):
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.elu(self.conv1(x, edge_index,edge_attr))
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index,edge_attr)
        return F.log_softmax(x, dim=1)

#create GAT model
num_classes =  3
edge_features = pyg_data.edge_attr.shape[1]
model = GAT(pyg_data.num_node_features, 16,num_classes,edge_features)

# Sanity checks
def test_check():
    print("=== Input Graph Shapes ===") #orginal shape
    print("x shape:", pyg_data.x.shape)
    print("edge_index shape:", pyg_data.edge_index.shape)
    print("edge_attr shape:", pyg_data.edge_attr.shape)

    model.eval()
    out = model(pyg_data,pyg_data.edge_attr.shape) #gat model shape



torch.Size([152, 5])


TypeError: dropout(): argument 'input' (position 1) must be Tensor, not Data

### Convert Network X graph to PyG dataset to train model

In [5]:
import torch
from torch_geometric.utils.convert import from_networkx
import torch.nn.functional as F
import torch_geometric.transforms as T

for node in G.nodes():
    G.nodes[node]["x"] = G.nodes[node]["mode_id"] 
    
# Convert to PyG
pyg_data = from_networkx(
    G,
    group_node_attrs=["x"],
    group_edge_attrs=["line_id", "length", "minutes", "transfer", "transfer_time"]
)

print(pyg_data)
print("Edge index shape:", pyg_data.edge_index.shape)
print("Edge attr shape:", pyg_data.edge_attr.shape)
print("num of node features:", pyg_data.num_node_features)

Data(edge_index=[2, 152], station_name=[75], mode_id=[75], transfer=[75], line_id=[75], x=[75, 1], edge_attr=[152, 5])
Edge index shape: torch.Size([2, 152])
Edge attr shape: torch.Size([152, 5])
num of node features: 1


### creating Trainning/Val/Test masks

Total nodes = 75

| Split      | Percentage | Nodes        |
| ---------- | ---------- | ------------ |
| Training   | **60%**    | **45 nodes** |
| Validation | **20%**    | **15 nodes** |
| Test       | **20%**    | **15 nodes** |


In [35]:
num_nodes = pyg_data.num_nodes

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

rand_perm = torch.randperm(num_nodes)

train_mask[rand_perm[:45]] = True
val_mask[rand_perm[45:60]] = True
test_mask[rand_perm[60:]] = True

pyg_data.train_mask = train_mask
pyg_data.val_mask = val_mask
pyg_data.test_mask = test_mask

print(f"num_train_nodes: {sum(train_mask)} | num_val_nodes: {sum(val_mask)} | num_test_nodes: {sum(test_mask)}")

num_train_nodes: 45 | num_val_nodes: 15 | num_test_nodes: 15


### Create GAT base model before trainning

In [31]:
from torch_geometric.nn import GATv2Conv

class GAT(torch.nn.Module):
    # in_channels - number of node feature
    # hidden_channels - 16
    # out_channels - number of classes output channels.set to 3, will rep congestion level to use in Dispact Agent layer
    # edge_features - number of edge feature
    def __init__(self, in_channels, hidden_channels, out_channels,edge_features):
        super().__init__()
        # First GAT layer with multi-head attention
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=8, edge_dim=edge_features)
        # Second GAT layer (output layer)
        self.conv2 = GATv2Conv(hidden_channels * 8,out_channels,heads=1,concat=False,edge_dim=edge_features)
        
    #dropout - default .6 or 60% once outr dataset is small and doent have alot of node feature
        #new default set to 0.1-0.2 for minimal regularization and small graphs
    def forward(self, x, edge_index, edge_attr):
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.elu(self.conv1(x, edge_index,edge_attr))
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index,edge_attr)
        return F.log_softmax(x, dim=1)

#create GAT model
num_classes =  3
edge_features = pyg_data.edge_attr.shape[1]
gat_model = GAT(pyg_data.num_node_features, 16,num_classes,edge_features)
optimizer = torch.optim.Adam(gat_model.parameters(), lr=0.005, weight_decay=5e-4)

In [32]:
print("=== Orginal datatypes ===")
print("x dtype:", pyg_data.x.dtype)
print("edge_index dtype:", pyg_data.edge_index.dtype)
print("edge_attr dtype:", pyg_data.edge_attr.dtype)

#convert to correct datatypes
pyg_data.x = pyg_data.x.float()
pyg_data.edge_attr = pyg_data.edge_attr.float()
pyg_data.edge_index = pyg_data.edge_index.long()

print("\n=== Corrected datatypes ===")
print("x dtype:", pyg_data.x.dtype)
print("edge_index dtype:", pyg_data.edge_index.dtype)
print("edge_attr dtype:", pyg_data.edge_attr.dtype)

=== Orginal datatypes ===
x dtype: torch.float32
edge_index dtype: torch.int64
edge_attr dtype: torch.float32

=== Corrected datatypes ===
x dtype: torch.float32
edge_index dtype: torch.int64
edge_attr dtype: torch.float32


In [33]:
# Sanity checks
def test_check():
    print("=== Input Graph Shapes ===") #orginal shape
    print("x shape:", pyg_data.x.shape)
    print("edge_index shape:", pyg_data.edge_index.shape)
    print("edge_attr shape:", pyg_data.edge_attr.shape)

    gat_model.eval()
    with torch.no_grad():
        out = gat_model(pyg_data.x,pyg_data.edge_index,pyg_data.edge_attr) #gat model shape without gradient change
        print("\n=== Model Output Shape ===")
        print("out shape:", out.shape)
    
test_check()


=== Input Graph Shapes ===
x shape: torch.Size([75, 1])
edge_index shape: torch.Size([2, 152])
edge_attr shape: torch.Size([152, 5])

=== Model Output Shape ===
out shape: torch.Size([75, 3])


### Create training/test eval functions

In [ ]:
def train():
    gat_model.train()
    optimizer.zero_grad()
    # Forward pass
    out = gat_model(pyg_data.x,pyg_data.edge_index,pyg_data.edge_attr)
    
    # Calculate loss on training nodes only
    loss = F.nll_loss(out[train_mask], pyg_data.y[train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

#test function to test the trainning accuracy
def test(model):
    model.eval()
    with torch.no_grad():
        #preiction on the whole graph
        logits = model(pyg_data.x,pyg_data.edge_index,pyg_data.edge_attr)
        
        #only focus on training_mask
        loss = F.nll_loss(logits[pyg_data.train_mask], pyg_data.y[pyg_data.train_mask])
        loss.backward()
        
        #compute accuracy
        preds = logits.argmax(dim=-1)
        acc = (preds[pyg_data.train_mask]==pyg_data.y[pyg_data.train_mask]).float().mean()
        
        return acc
    
# funct to vaildate model
def evaluate(model):
    model.eval()
    with torch.no_grad():
        
        #forward
        logits = model(pyg_data.x,pyg_data.edge_index,pyg_data.edge_attr)
        loss = F.nll_loss(logits[pyg_data.train_mask], pyg_data.y[pyg_data.train_mask])
        
        #compute accuracy
        preds = logits.argmax(dim=-1) #pred = logits[mask].max(1)[1]
        acc = (preds[pyg_data.val_mask]==pyg_data.y[pyg_data.val_mask]).float().mean()
        
        return loss, acc     

epoch_num = 40

# Training loop over epochs
for epoch in range(epoch_num):
    train_loss = train()
    train_acc = test(gat_model)
    val_loss, val_acc = evaluate(gat_model)
    
    if epoch%5==0 or epoch == epoch_num-1:
        print(f"Epoch: {epoch} | train_loss: {train_loss:.4f} | train_acc: {train_acc*100:.2f}% |" f" val_loss: {val_loss:.4f} | val_acc: {val_acc*100:.2f}%")
        


TypeError: 'NoneType' object is not subscriptable